# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# ---------- Feature Scaling ----------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train only
X_test_s  = scaler.transform(X_test)

# ---------- Feature Selection ----------
# pick k that is <= number of features (e.g., top-10 or all if fewer)
k = min(10, X_train.shape[1])   # change 10 to what you want
selector = SelectKBest(score_func=f_classif, k=k)

X_train_sel = selector.fit_transform(X_train_s, y_train)
X_test_sel  = selector.transform(X_test_s)

# names of selected features
selected_features = X.columns[selector.get_support()]

print("Scaled shapes :", X_train_s.shape, X_test_s.shape)
print(f"Selected top-{k} features ({len(selected_features)}):")
print(selected_features.tolist())
print("Final shapes  :", X_train_sel.shape, X_test_sel.shape)


Scaled shapes : (5284, 19) (1322, 19)
Selected top-10 features (10):
['RoomService', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'CryoSleep_True', 'Destination_TRAPPIST-1e', 'Deck_B', 'Deck_C', 'Deck_E', 'Deck_F']
Final shapes  : (5284, 10) (1322, 10)


**Perform Train Test Split**

In [7]:
#your code here
from sklearn.model_selection import train_test_split

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(5284, 19) (1322, 19) (5284,) (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [8]:
#your code here
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np
import os

# Use your selected feature set if available; else fall back to raw split
try:
    X_tr, X_te = X_train_sel, X_test_sel
except NameError:
    X_tr, X_te = X_train, X_test
y_tr, y_te = y_train, y_test

rng = 42
n_jobs = min(8, os.cpu_count() or 1)
base_tree = DecisionTreeClassifier(random_state=rng)

# ---- Bagging (bootstrap = True) ----
bag_clf = BaggingClassifier(
    estimator=base_tree,
    n_estimators=300,
    max_samples=0.8,      # 80% of rows per bootstrap sample
    max_features=1.0,     # all features
    bootstrap=True,       # with replacement -> Bagging
    bootstrap_features=False,
    oob_score=True,       # OOB score only valid in bagging
    n_jobs=n_jobs,
    random_state=rng
)
bag_clf.fit(X_tr, y_tr)

# ---- Pasting (bootstrap = False) ----
past_clf = BaggingClassifier(
    estimator=base_tree,
    n_estimators=300,
    max_samples=0.8,      # 80% of rows per paste sample
    max_features=1.0,
    bootstrap=False,      # without replacement -> Pasting
    bootstrap_features=False,
    n_jobs=n_jobs,
    random_state=rng
)
past_clf.fit(X_tr, y_tr)

# ---- Evaluate helper ----
def eval_model(name, clf):
    y_pred = clf.predict(X_te)
    # Trees have predict_proba; guard for safety
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, y_proba)
    else:
        auc = np.nan
    acc = accuracy_score(y_te, y_pred)
    print(f"\n=== {name} ===")
    if hasattr(clf, "oob_score_"):
        print(f"OOB score: {clf.oob_score_:.4f}")
    print(f"Accuracy : {acc:.4f}")
    if not np.isnan(auc):
        print(f"ROC AUC  : {auc:.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_te, y_pred))
    print("Classification report:\n", classification_report(y_te, y_pred, digits=4))

# ---- Reports ----
eval_model("Bagging (bootstrap=True)", bag_clf)
eval_model("Pasting (bootstrap=False)", past_clf)



=== Bagging (bootstrap=True) ===
OOB score: 0.7793
Accuracy : 0.7648
ROC AUC  : 0.8504
Confusion matrix:
 [[460 196]
 [115 551]]
Classification report:
               precision    recall  f1-score   support

           0     0.8000    0.7012    0.7474       656
           1     0.7376    0.8273    0.7799       666

    accuracy                         0.7648      1322
   macro avg     0.7688    0.7643    0.7636      1322
weighted avg     0.7686    0.7648    0.7638      1322


=== Pasting (bootstrap=False) ===
Accuracy : 0.7526
ROC AUC  : 0.8374
Confusion matrix:
 [[449 207]
 [120 546]]
Classification report:
               precision    recall  f1-score   support

           0     0.7891    0.6845    0.7331       656
           1     0.7251    0.8198    0.7696       666

    accuracy                         0.7526      1322
   macro avg     0.7571    0.7521    0.7513      1322
weighted avg     0.7569    0.7526    0.7514      1322



- Random Forests

In [9]:
#your code here
# ==== Random Forests ====
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Use selected features if defined; otherwise fall back to full set
try:
    X_tr, X_te = X_train_sel, X_test_sel
    feat_names = (X.columns[selector.get_support()]).tolist()
except NameError:
    X_tr, X_te = X_train, X_test
    feat_names = list(X_train.columns)

rng = 42
n_jobs = min(8, os.cpu_count() or 1)

# --- quick baseline RF ---
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    oob_score=True,
    n_jobs=n_jobs,
    random_state=rng,
)
rf.fit(X_tr, y_train)

y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

print("Baseline RF")
print(f"OOB score : {rf.oob_score_:.4f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred, digits=4))

# --- hyperparameter tuning (small grid) ---
param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [None, 12, 24],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.5],  # try sqrt or half of features
}

gs = GridSearchCV(
    RandomForestClassifier(
        oob_score=True, n_jobs=n_jobs, random_state=rng
    ),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=n_jobs,
    refit=True,
    verbose=0,
)
gs.fit(X_tr, y_train)
best_rf = gs.best_estimator_

yp = best_rf.predict(X_te)
pp = best_rf.predict_proba(X_te)[:, 1]

print("\nTuned RF")
print("Best params:", gs.best_params_)
print(f"CV best AUC: {gs.best_score_:.4f}")
print(f"Accuracy   : {accuracy_score(y_test, yp):.4f}")
print(f"ROC AUC    : {roc_auc_score(y_test, pp):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, yp))
print("Classification report:\n", classification_report(y_test, yp, digits=4))

# --- feature importance (top 15) ---
importances = pd.Series(best_rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print("\nTop 15 features:\n", importances.head(15))


Baseline RF
OOB score : 0.7782
Accuracy  : 0.7587
ROC AUC   : 0.8452
Confusion matrix:
 [[455 201]
 [118 548]]
Classification report:
               precision    recall  f1-score   support

           0     0.7941    0.6936    0.7404       656
           1     0.7316    0.8228    0.7746       666

    accuracy                         0.7587      1322
   macro avg     0.7629    0.7582    0.7575      1322
weighted avg     0.7626    0.7587    0.7576      1322


Tuned RF
Best params: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
CV best AUC: 0.8732
Accuracy   : 0.7769
ROC AUC    : 0.8626
Confusion matrix:
 [[470 186]
 [109 557]]
Classification report:
               precision    recall  f1-score   support

           0     0.8117    0.7165    0.7611       656
           1     0.7497    0.8363    0.7906       666

    accuracy                         0.7769      1322
   macro avg     0.7807    0.7764    0.7759      1322
weighte

- Gradient Boosting

In [12]:
#your code here
# ============================
# Full, updated end-to-end code
# (Spaceship Titanic – ensembles + tidy eval)
# ============================

import os
import warnings
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")
os.environ["LOKY_MAX_CPU_COUNT"] = str(min(8, os.cpu_count() or 1))

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)

# --- 1) Load & basic cleaning ---
spaceship = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
)

# Drop any missing rows (as per earlier steps)
df = spaceship.dropna().reset_index(drop=True)

# Cabin -> Deck letter, drop granularity + id/name
df["Deck"] = df["Cabin"].str.split("/").str[0]
df = df.drop(columns=["PassengerId", "Name", "Cabin"], errors="ignore")

# Target to numeric 0/1
df["Transported"] = df["Transported"].astype(int)

# Booleans -> ints (KNN/metrics friendly)
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

# One-hot encode categoricals (exclude target)
cat_cols = df.select_dtypes(include=["object", "category"]).columns.drop("Transported", errors="ignore")
X = pd.get_dummies(df.drop(columns="Transported"), columns=cat_cols, drop_first=True)
y = df["Transported"]

# --- 2) Train / test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Shapes ->",
      f"X_train: {X_train.shape}, X_test: {X_test.shape},",
      f"y_train: {y_train.shape}, y_test: {y_test.shape}")

# --- 3) Feature scaling (kept for completeness; tree ensembles don’t need it) ---
scaler = StandardScaler(with_mean=False)  # with_mean=False keeps sparse matrices safe if any
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# --- 4) Feature selection (ANOVA F) ---
k = min(15, X_train.shape[1])  # choose <= number of features
selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X_train, y_train)
selected_features = X_train.columns[selector.get_support()]
X_train_sel = X_train[selected_features].copy()
X_test_sel  = X_test[selected_features].copy()

print(f"Selected {len(selected_features)} features:",
      list(selected_features)[:12], "..." if len(selected_features) > 12 else "")

# ============================
# Helper: concise evaluation
# ============================
def evaluate_model(name, clf, X_te, y_te):
    """Print Accuracy, AUC, CM and a compact report; return metrics dict."""
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_te)[:, 1]
    else:
        y_proba = clf.decision_function(X_te)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC AUC  : {auc:.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_te, y_pred))
    print(pd.DataFrame(classification_report(y_te, y_pred, digits=4, output_dict=True)).T)

    return {"model": name, "accuracy": acc, "auc": auc, "proba": y_proba, "pred": y_pred}

# ============================
# 5) HistGradientBoosting (baseline + tuned)
# ============================
from sklearn.ensemble import HistGradientBoostingClassifier

rng = 42
hgb_base = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    max_iter=300,
    early_stopping=True,
    l2_regularization=0.1,
    random_state=rng
)
hgb_base.fit(X_train_sel, y_train)
res_hgb_base = evaluate_model("HistGradientBoosting (baseline)", hgb_base, X_test_sel, y_test)

hgb_grid = {
    "learning_rate": [0.05, 0.1],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [10, 20, 50],
    "l2_regularization": [0.0, 0.1, 1.0],
    "max_iter": [200, 300],
    "max_depth": [None, 6],
}
hgb_gs = GridSearchCV(
    HistGradientBoostingClassifier(early_stopping=True, random_state=rng),
    param_grid=hgb_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=min(8, os.cpu_count() or 1),
    refit=True,
    verbose=0,
)
hgb_gs.fit(X_train_sel, y_train)
best_hgb = hgb_gs.best_estimator_
print("\n[HGB] Best params:", hgb_gs.best_params_)
print("[HGB] CV best AUC:", f"{hgb_gs.best_score_:.4f}")
res_hgb_tuned = evaluate_model("HistGradientBoosting (tuned)", best_hgb, X_test_sel, y_test)

# ============================
# 6) Classic Gradient Boosting (baseline + tuned)
# ============================
from sklearn.ensemble import GradientBoostingClassifier

gb_base = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=rng
)
gb_base.fit(X_train_sel, y_train)
res_gb_base = evaluate_model("GradientBoosting (baseline)", gb_base, X_test_sel, y_test)

gb_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
}
gb_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=rng),
    param_grid=gb_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=n_jobs,
    refit=True,
    verbose=0
)
gb_gs.fit(X_tr, y_train)
best_gb = gb_gs.best_estimator_
print("\n[GB] Best params:", gb_gs.best_params_)
print("[GB] CV best AUC:", f"{gb_gs.best_score_:.4f}")
evaluate("GradientBoosting (tuned)", best_gb)


Shapes -> X_train: (5284, 19), X_test: (1322, 19), y_train: (5284,), y_test: (1322,)
Selected 15 features: ['Age', 'RoomService', 'FoodCourt', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'CryoSleep_True', 'Destination_TRAPPIST-1e', 'VIP_True', 'Deck_B', 'Deck_C', 'Deck_D'] ...

=== HistGradientBoosting (baseline) ===
Accuracy : 0.7753
ROC AUC  : 0.8698
Confusion matrix:
 [[484 172]
 [125 541]]
              precision    recall  f1-score     support
0              0.794745  0.737805  0.765217   656.00000
1              0.758766  0.812312  0.784627   666.00000
accuracy       0.775340  0.775340  0.775340     0.77534
macro avg      0.776756  0.775059  0.774922  1322.00000
weighted avg   0.776620  0.775340  0.774995  1322.00000

[HGB] Best params: {'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
[HGB] CV best AUC: 0.8830

=== HistGradientBoosting (tuned) ===
Accuracy : 0.8026
ROC AUC  : 0.8794
Confusion matrix:
 [[492

- Adaptive Boosting

In [14]:
# --- AdaBoost (SAMME) updated: no deprecation warning ---
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Use your selected features if you have them; else fall back to full split
try:
    X_tr, X_te = X_train_sel, X_test_sel
except NameError:
    X_tr, X_te = X_train, X_test
y_tr, y_te = y_train, y_test

rng = 42
n_jobs = min(8, os.cpu_count() or 1)

# Detect estimator parameter name for compatibility
try:
    AdaBoostClassifier(estimator=DecisionTreeClassifier())  # sklearn >= 1.2
    ada_base = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=rng),
        algorithm="SAMME",
        n_estimators=400,
        learning_rate=0.1,
        random_state=rng,
    )
    depth_key = "estimator__max_depth"
except TypeError:
    # Older sklearn: uses base_estimator
    ada_base = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=rng),
        algorithm="SAMME",
        n_estimators=400,
        learning_rate=0.1,
        random_state=rng,
    )
    depth_key = "base_estimator__max_depth"

# Fit baseline
ada_base.fit(X_tr, y_tr)

# Evaluate helper (SAMME may not expose predict_proba; fall back to decision_function)
def eval_clf(name, clf):
    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(X_te)[:, 1]
    else:
        scores = clf.decision_function(X_te)
    preds = clf.predict(X_te)
    print(f"\n=== {name} ===")
    print(f"Accuracy : {accuracy_score(y_te, preds):.4f}")
    print(f"ROC AUC  : {roc_auc_score(y_te, scores):.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_te, preds))
    print(pd.DataFrame(classification_report(y_te, preds, digits=4, output_dict=True)).T)

eval_clf("AdaBoost (SAMME) baseline", ada_base)

# Small grid search
param_grid = {
    "n_estimators": [200, 400, 600],
    "learning_rate": [0.05, 0.1, 0.2, 0.5],
    depth_key: [1, 2, 3],
}

gs = GridSearchCV(
    estimator=ada_base,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=n_jobs,
    refit=True,
    verbose=0,
)
gs.fit(X_tr, y_tr)
best_ada = gs.best_estimator_

print("\n[AdaBoost SAMME] Best params:", gs.best_params_)
print("[AdaBoost SAMME] CV best AUC:", f"{gs.best_score_:.4f}")

eval_clf("AdaBoost (SAMME) tuned", best_ada)



=== AdaBoost (SAMME) baseline ===
Accuracy : 0.7413
ROC AUC  : 0.8374
Confusion matrix:
 [[556 100]
 [242 424]]
              precision    recall  f1-score      support
0              0.696742  0.847561  0.764787   656.000000
1              0.809160  0.636637  0.712605   666.000000
accuracy       0.741301  0.741301  0.741301     0.741301
macro avg      0.752951  0.742099  0.738696  1322.000000
weighted avg   0.753376  0.741301  0.738499  1322.000000

[AdaBoost SAMME] Best params: {'estimator__max_depth': 3, 'learning_rate': 0.5, 'n_estimators': 400}
[AdaBoost SAMME] CV best AUC: 0.8795

=== AdaBoost (SAMME) tuned ===
Accuracy : 0.7761
ROC AUC  : 0.8652
Confusion matrix:
 [[479 177]
 [119 547]]
              precision    recall  f1-score      support
0              0.801003  0.730183  0.763955   656.000000
1              0.755525  0.821321  0.787050   666.000000
accuracy       0.776097  0.776097  0.776097     0.776097
macro avg      0.778264  0.775752  0.775503  1322.000000
weighted av

Which model is the best and why?

In [16]:
#comment here

Best ranking power: highest AUC ≈ 0.8701 (CV AUC ≈ 0.8751 → very close to test → stable generalization).

Good class-1 recall: ~0.863 (from your confusion matrix 575/666), which is valuable if catching “Transported=1” matters.

Robust to one-hot/sparse, nonlinear interactions: tree boosting handles your engineered features without scaling and is fast at prediction.